# 훅을 사용한 Strands 에이전트와 AgentCore 메모리 튜토리얼

## 개요

이 튜토리얼에서는 훅을 통해 AgentCore 메모리와 통합된 Strands 에이전트를 사용하여 지능형 개인 비서를 구축하는 방법을 시연합니다. 에이전트는 대화 컨텍스트를 유지하고 상호작용에서 학습하여 개인화된 응답을 제공합니다.

## 튜토리얼 세부사항

**사용 사례**: 수학 어시스턴트

| 정보                | 세부사항                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형        | 장기 대화형                                                                       |
| 에이전트 유형        | 수학 어시스턴트                                                                   |
| 에이전트 프레임워크   | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소   | 메모리를 위한 AgentCore 요약 전략, 메모리 저장 및 조회를 위한 훅                      |
| 예제 난이도          | 중급                                                                              |


학습 내용:
- 대화 요약으로 AgentCore 메모리 설정
- 자동 저장 및 조회를 위한 메모리 훅 생성
- 영구 메모리를 갖춘 Strands 에이전트 구축
- 대화 간 메모리 기능 테스트
- 대안적 학습 경로를 위한 대화 브랜칭 사용
- 학생 진도 및 성과 추적을 위한 메타데이터 적용

### 시나리오 컨텍스트

이 예제에서는 이전 대화의 요약을 저장하는 수학 어시스턴트 예제를 만들게 됩니다.
이 예제의 주요 기능:
- **자동 메모리 저장**: 대화가 자동으로 저장됩니다
- **컨텍스트 조회**: 이전 대화가 현재 응답에 참조됩니다
- **요약 생성**: 핵심 정보가 추출되고 요약됩니다
- **도구 통합**: 수학 연산을 위한 계산기 도구
- **대화 브랜칭**: 대안적인 난이도 수준과 교수법 탐색
- **메타데이터 추적**: 난이도, 성과, 학습 마일스톤으로 이벤트 태깅

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
- Python 3.10+
- Amazon Bedrock AgentCore 메모리 권한이 있는 AWS 자격 증명
- Amazon Bedrock AgentCore SDK

## 1단계: 환경 설정
필요한 모든 라이브러리를 임포트하고 이 노트북을 작동시키기 위한 클라이언트를 정의하는 것부터 시작하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.memory.manager import Memory, MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import (
    SemanticStrategy
)
from bedrock_agentcore.memory import MemorySessionManager
from bedrock_agentcore.memory.constants import (
    ConversationalMessage, MessageRole, RetrievalConfig
)
from bedrock_agentcore.memory.models import (
    StringValue, EventMetadataFilter, LeftExpression, RightExpression, OperatorType
)


In [ ]:
import os
import logging
from typing import List, Dict
from strands import Agent
from datetime import datetime
from strands_tools import calculator
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

# 로깅 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("memory-tutorial")

# 설정 - 사용자의 값으로 교체하세요
REGION = os.getenv('AWS_REGION', 'us-west-2')
ACTOR_ID = f"student-{datetime.now().strftime('%Y%m%d%H%M%S')}"
SESSION_ID = f"math-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

# 코드 간결성을 위한 메시지 역할 상수 정의
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

## 2단계: 메모리 리소스 생성

이 단계에서는 시맨틱 전략으로 메모리 리소스를 생성합니다. 이 리소스는 대화 데이터를 저장하고 구성합니다. 내장 SemanticStrategy는 IAM 실행 역할 없이도 대화에서 자동으로 사실 정보를 캡처합니다.

In [ ]:
from botocore.exceptions import ClientError

# MemoryManager 초기화
memory_manager = MemoryManager(region_name=REGION)
memory_name = "MathAssistant"

# SemanticStrategy를 사용하여 메모리 전략 정의
strategies = [
    SemanticStrategy(
        name="MathLearningMemory",
        description="Captures facts from math learning conversations",
        namespaces=["/students/math/{actorId}/"]
    )
]

# MemoryManager를 사용하여 메모리 리소스 생성
memory_id = None  # 예외 핸들러에서 NameError를 방지하기 위해 초기화
try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=strategies,
        description="Memory for tutorial agent",
        event_expiry_days=30
    )
    memory_id = memory.id
    logger.info(f"메모리가 생성되었습니다: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 처리
    logger.error(f"오류: {e}")
    import traceback
    traceback.print_exc()
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            memory_manager.delete_memory(memory_id)
            logger.info(f"메모리가 정리되었습니다: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"메모리 정리 실패: {cleanup_error}")
    # 실행을 중단하기 위해 예외를 다시 발생시킴
    raise

# memory_id가 성공적으로 획득되었는지 확인
if memory_id is None:
    raise RuntimeError("메모리 ID 생성 또는 조회에 실패했습니다")

## 3단계: 세션 매니저 초기화

이제 학생을 위한 MemorySessionManager와 MemorySession을 생성합니다. 세션 매니저는 모든 작업에서 memory_id, actor_id, session_id 파라미터를 자동으로 처리하여 더 깔끔한 API를 제공합니다.

이 세션 기반 접근 방식은 메모리 작업을 간소화하고 코드를 더 유지보수하기 쉽게 만듭니다.

In [ ]:
# 세션 매니저 초기화
session_manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)

# 특정 학생에 대한 메모리 세션 생성
student_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID,
    session_id=SESSION_ID
)

logger.info(f"메모리 {memory_id}에 대한 세션 매니저가 초기화되었습니다")
logger.info(f"액터 {ACTOR_ID}에 대한 학생 세션이 생성되었습니다")
logger.info(f"   세션 ID: {SESSION_ID}")

## 4단계: 세션 지원 메모리 훅 프로바이더 생성

이 단계에서는 MemorySession을 사용하여 메모리 작업을 자동화하는 커스텀 `MemoryHookProvider` 클래스를 정의합니다. 훅은 에이전트 실행 수명 주기의 특정 시점에서 실행되는 특수 함수입니다. 우리가 만드는 메모리 훅은 두 가지 주요 기능을 수행합니다:

1. **메모리 조회**: `search_long_term_memories()`를 사용하여 사용자가 메시지를 보낼 때 관련 과거 대화를 자동으로 가져옴
2. **메모리 저장**: `add_turns()`를 사용하여 에이전트가 응답한 후 ConversationalMessage 객체로 새로운 대화를 저장

이를 통해 수동 관리 없이도 매끄러운 메모리 경험이 제공되며, 세션 기반 API를 통해 memory_id, actor_id, session_id를 반복적으로 전달할 필요가 없어집니다.

In [ ]:
class MemoryHookProvider(HookProvider):
    """MemorySession을 사용한 자동 메모리 관리 훅 프로바이더"""
    
    def __init__(self, student_session):
        """MemorySession 인스턴스로 초기화
        
        Args:
            student_session: 학생을 위한 MemorySession 인스턴스
        """
        self.student_session = student_session
        
        # 수학 학습 컨텍스트를 위한 조회 구성 정의
        self.retrieval_config = RetrievalConfig(
            top_k=5,
            relevance_score=0.3
        )
    
    def retrieve_memories(self, event: MessageAddedEvent):
        """MemorySession을 사용하여 사용자 메시지 처리 전 관련 메모리 조회"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_message = messages[-1]["content"][0].get("text", "")
            
            try:
                # MemorySession을 사용하여 컨텍스트 조회 (actor_id 전달 불필요)
                namespace_prefix = f"/students/math/{self.student_session._actor_id}/"
                
                # 세션 API를 사용하여 장기 메모리 검색
                memories = self.student_session.search_long_term_memories(
                    query=user_message,
                    namespace_prefix=namespace_prefix,
                    top_k=self.retrieval_config.top_k
                )
                
                # 관련성 점수로 필터링
                filtered_memories = [
                    memory for memory in memories
                    if memory.get("score", 0) >= self.retrieval_config.relevance_score
                ]
                
                # 메모리 콘텐츠 추출
                memory_context = []
                for memory in filtered_memories:
                    if isinstance(memory, dict):
                        content = memory.get('content', {})
                        if isinstance(content, dict):
                            text = content.get('text', '').strip()
                            score = memory.get('score', 0)
                            if text:
                                memory_context.append(f"[Score: {score:.2f}] {text}")
                
                # 메모리를 사용자 메시지에 주입
                if memory_context:
                    context_text = "\n".join(memory_context)
                    original_text = messages[-1]["content"][0].get("text", "")
                    messages[-1]["content"][0]["text"] = (
                        f"{original_text}\n\nStudent Learning Context:\n{context_text}"
                    )
                    logger.info(f"{len(memory_context)}개의 관련 메모리를 조회했습니다 (전체 {len(memories)}개에서 필터링)")
                    
            except Exception as e:
                logger.error(f"메모리 조회 실패: {e}")
    
    def save_memories(self, event: AfterInvocationEvent):
        """MemorySession을 사용하여 에이전트 응답 후 대화 저장"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # 마지막 사용자 및 어시스턴트 메시지 가져오기
                user_msg = None
                assistant_msg = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not assistant_msg:
                        assistant_msg = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_msg and "toolResult" not in msg["content"][0]:
                        user_msg = msg["content"][0]["text"]
                        break
                
                if user_msg and assistant_msg:
                    # ConversationalMessage 객체를 사용하여 MemorySession으로 저장
                    interaction_messages = [
                        ConversationalMessage(user_msg, USER),
                        ConversationalMessage(assistant_msg, ASSISTANT)
                    ]
                    
                    result = self.student_session.add_turns(interaction_messages)
                    logger.info(f"MemorySession을 사용하여 대화가 저장되었습니다 - 이벤트 ID: {result['eventId']}")
                    
        except Exception as e:
            logger.error(f"메모리 저장 실패: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """메모리 훅 등록"""
        registry.add_callback(MessageAddedEvent, self.retrieve_memories)
        registry.add_callback(AfterInvocationEvent, self.save_memories)
        logger.info("MemorySession 지원으로 메모리 훅이 등록되었습니다")

## 5단계: 메모리를 갖춘 에이전트 생성

이제 MemorySession을 사용하는 메모리 훅 프로바이더와 Strands 에이전트를 생성하고 연결합니다. 이 에이전트는 두 가지 핵심 기능을 갖습니다:

1. **메모리 통합**: 생성한 메모리 훅이 세션 기반 작업을 통해 자동 컨텍스트 조회를 활성화합니다
2. **계산기 도구**: 에이전트가 필요시 수학 연산을 수행할 수 있습니다

이 조합은 학생의 진도를 기억하면서 유용한 계산을 수행할 수 있는 수학 튜터를 만듭니다.

In [ ]:
# MemorySession으로 메모리 훅 프로바이더 생성
memory_hooks = MemoryHookProvider(student_session)

# 메모리 훅과 계산기 도구로 에이전트 생성
agent = Agent(
    hooks=[memory_hooks],
    model = "global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[calculator],
    # 시스템 프롬프트: 도움이 되는 개인 수학 튜터. 수학 문제 해결을 돕고 학습 진도와 선호도에 기반한 개인화된 지원을 제공합니다.
    system_prompt="You are a helpful personal math tutor. You assist users in solving math problems and provide personalized assistance based on their learning progress and preferences.",
)

logger.info("MemorySession 기반 훅으로 에이전트가 생성되었습니다")
logger.info(f"   학생: {ACTOR_ID}")
logger.info(f"   세션: {SESSION_ID}")

**에이전트가 설정되었습니다! 이제 테스트해 보겠습니다.**

## 메모리 기능 테스트

이 섹션에서는 일련의 상호작용을 통해 에이전트의 메모리 기능을 테스트합니다. 에이전트가 시간이 지남에 따라 어떻게 컨텍스트를 구축하고 이전 상호작용을 회상하는지 관찰합니다.

먼저 에이전트에게 자기 소개를 하고 수학 질문을 해보겠습니다:

In [ ]:
# 첫 번째 상호작용 - 자기소개
# "안녕하세요, 저는 John이고 이산수학 과정에 막 등록했습니다. 이 문제를 풀어주세요: 5권의 책을 선반에 배열하는 방법은 몇 가지인가요?"
response1 = agent("Hi, I'm John and I just enrolled in Discrete Math course. Help me solve this: How many ways can I arrange 5 books on a shelf?")
print(f"에이전트: {response1}")

에이전트에게 또 다른 계산 작업을 주어 보겠습니다:

In [ ]:
# 두 번째 상호작용 - 또 다른 계산
# "단계별 설명과 예제 문제를 통해 더 잘 배웁니다. 모듈러 산술을 설명해 주세요. 17 mod 5는 뭐예요?"
response2 = agent("I learn better with step-by-step explanation with example questions. Can you explain modular arithmetic? What's 17 mod 5?")
print(f"에이전트: {response2}")

이제 에이전트가 우리가 누구인지 기억하는지 확인해 보겠습니다.

**참고:** 메모리가 추출, 통합 및 저장되는 데 시간이 필요하므로 여기서 약 20초 정도 기다려 주세요.

In [ ]:
# 세 번째 상호작용 - 메모리 기억 테스트
# "맞았습니다! 모듈러 산술 다음에 공부해야 할 즉각적인 다음 단계는 무엇인가요?"
response3 = agent("I got that right! What's the immediate next step that I should study after modular arithmetic?")
print(f"에이전트: {response3}")

마지막으로, 에이전트가 이전 계산 기록을 기억하는지 확인해 보겠습니다:

In [ ]:
# 네 번째 상호작용 - 컨텍스트 인식 테스트
# "이건 너무 어려워요, 더 쉬운 걸 해볼 수 있을까요?"
response4 = agent("This is too hard, can we try something easier?")
print(f"에이전트: {response4}")

### 메모리 저장 확인

마지막 단계로, 대화가 AgentCore 메모리에 올바르게 저장되었는지 확인합니다. 이를 통해 메모리 훅이 올바르게 작동하고 에이전트가 향후 상호작용에서 이 정보에 접근할 수 있음을 보여줍니다.

In [ ]:
# MemorySession을 사용하여 저장된 메모리 확인
try:
    namespace_prefix = f"/students/math/{ACTOR_ID}/"
    
    memories = student_session.search_long_term_memories(
        query="mathematics calculations learning progress",
        namespace_prefix=namespace_prefix,
        top_k=5
    )
    
    print(f"\n학생 {ACTOR_ID}의 메모리 {len(memories)}개를 찾았습니다:")
    print("=" * 60)
    for i, memory in enumerate(memories, 1):
        if isinstance(memory, dict):
            content = memory.get('content', {})
            score = memory.get('score', 0)
            if isinstance(content, dict):
                text = content.get('text', '')[:200] + "..."
                print(f"\n{i}. [관련성: {score:.2f}]")
                print(f"   {text}")
    print("\n" + "=" * 60)
                
except Exception as e:
    logger.error(f"메모리 조회 오류: {e}")

## 고급 기능: 브랜칭 및 메타데이터

### 대화 브랜칭

브랜칭을 사용하면 어떤 시점에서든 대안적인 대화 경로를 탐색할 수 있습니다. 이는 다음과 같은 경우에 유용합니다:
- 다양한 난이도 수준 테스트
- 대안적인 설명 탐색
- 교수법 A/B 테스트

더 어려운 문제 경로를 탐색하기 위한 브랜치를 만들어 보겠습니다:

In [ ]:
# 대화에서 마지막 이벤트 ID 가져오기
events = student_session.list_events()
if events:
    last_event_id = events[-1].eventId
    
    # 고급 주제를 탐색하기 위해 대화 분기
    branch_event = student_session.fork_conversation(
        root_event_id=last_event_id,
        branch_name="advanced-path",
        messages=[
            ConversationalMessage(
                "Actually, I'm ready for a challenge! Can you give me a harder problem involving modular arithmetic and combinatorics?",
                USER
            ),
            ConversationalMessage(
                "Great! Here's a challenging problem: How many 4-digit numbers are there where the sum of digits is congruent to 3 (mod 5)? This combines modular arithmetic with counting principles.",
                ASSISTANT
            )
        ]
    )
    
    logger.info(f"이벤트 {last_event_id}에서 'advanced-path' 브랜치가 생성되었습니다")
    logger.info(f"   브랜치 이벤트 ID: {branch_event['eventId']}")
    
    # 모든 브랜치 목록 조회
    branches = student_session.list_branches()
    print(f"\n세션에 {len(branches)}개의 브랜치가 있습니다:")
    for branch in branches:
        print(f"   - {branch.name}: {branch.event_count}개의 이벤트")
    
    # 고급 브랜치의 이벤트 가져오기
    advanced_events = student_session.list_events(branch_name="advanced-path")
    print(f"\n고급 브랜치에 {len(advanced_events)}개의 이벤트가 있습니다")
else:
    print("브랜치를 생성할 이벤트가 없습니다")

### 메타데이터 활용

메타데이터를 사용하면 더 나은 정리와 조회를 위해 이벤트에 커스텀 정보를 태깅할 수 있습니다. 메타데이터를 사용하여 다음을 추적해 보겠습니다:
- 문제 난이도 수준
- 학생 성과
- 주제 카테고리
- 학습 마일스톤

In [ ]:
from bedrock_agentcore.memory.models import StringValue

# 풍부한 메타데이터와 함께 새로운 상호작용 추가
metadata_event = student_session.add_turns(
    messages=[
        ConversationalMessage(
            "Let me try: If I choose 3 books from 5, that's C(5,3) = 10 ways, right?",
            USER
        ),
        ConversationalMessage(
            "Excellent! You correctly applied the combination formula. That's exactly right: C(5,3) = 5!/(3!×2!) = 10.",
            ASSISTANT
        )
    ],
    metadata={
        "difficulty": StringValue.build("intermediate"),
        "topic": StringValue.build("combinatorics"),
        "subtopic": StringValue.build("combinations"),
        "performance": StringValue.build("correct"),
        "milestone": StringValue.build("first_correct_combination"),
        "learning_stage": StringValue.build("applying_formulas")
    }
)

logger.info(f"메타데이터가 포함된 이벤트 추가 완료 - 이벤트 ID: {metadata_event['eventId']}")
print("\n이벤트 태그 정보:")
print("   - 난이도: intermediate")
print("   - 주제: combinatorics")
print("   - 성과: correct")
print("   - 마일스톤: first_correct_combination")

### 메타데이터별 이벤트 쿼리

이제 메타데이터를 기반으로 이벤트를 필터링하여 학생 진도를 분석할 수 있습니다:

In [ ]:
from bedrock_agentcore.memory.models import EventMetadataFilter, LeftExpression, RightExpression, OperatorType

# 학생이 정답을 맞힌 이벤트 쿼리
try:
    correct_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "performance"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "correct"}}
            }
        ]
    )
    
    print(f"\n학생이 정답을 맞힌 이벤트 {len(correct_events)}개를 찾았습니다")
    
    # 중급 난이도 문제 쿼리
    intermediate_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "difficulty"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "intermediate"}}
            }
        ]
    )
    
    print(f"중급 난이도 문제 {len(intermediate_events)}개를 찾았습니다")
    
    # 조합론 주제 쿼리
    combinatorics_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "topic"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "combinatorics"}}
            }
        ]
    )
    
    print(f"조합론 관련 이벤트 {len(combinatorics_events)}개를 찾았습니다")
    
    print("\n메타데이터 활용 사례:")
    print("   - 난이도별 학생 진도 추적")
    print("   - 더 많은 연습이 필요한 주제 식별")
    print("   - 성과 보고서 생성")
    print("   - 이력 기반 학습 경로 개인화")
    
except Exception as e:
    logger.error(f"메타데이터 쿼리 오류: {e}")
    print(f"참고: 메타데이터 필터링에는 메타데이터 태그가 있는 이벤트가 필요합니다")

튜토리얼이 완료되었습니다!

주요 내용 정리:
- 메모리 훅은 대화 컨텍스트를 자동으로 저장하고 조회합니다
- 에이전트는 여러 상호작용에 걸쳐 상태를 유지할 수 있습니다
- AgentCore 메모리는 관련 컨텍스트에 대한 시맨틱 검색을 제공합니다
- 도구를 메모리와 결합하여 향상된 기능을 제공할 수 있습니다
- **브랜칭을 통해 대안적인 대화 경로를 탐색할 수 있습니다**
- **메타데이터는 강력한 필터링 및 분석 기능을 제공합니다**

## 정리

### 선택 사항: 메모리 리소스 삭제

튜토리얼을 완료한 후, 불필요한 비용이 발생하지 않도록 메모리 리소스를 삭제할 수 있습니다. 다음 코드는 정리를 위해 제공되지만 기본적으로 주석 처리되어 있습니다.

In [ ]:
# 메모리 리소스를 삭제하려면 주석을 해제하세요
# try:
#     memory_manager.delete_memory(memory_id)
#     print(f"메모리 리소스가 삭제되었습니다: {memory_id}")
# except Exception as e:
#     print(f"메모리 삭제 오류: {e}")